# 生成经验回放数据（防止灾难性遗忘）

用 Qwen3-4B 自身生成 30 条通用问答数据，微调时混入训练集，让模型不会忘记基础能力。

**原理：** 微调时只训练角色扮演数据会导致模型遗忘通用知识。混入一批由模型自己生成的通用问答，相当于告诉模型「这些回答我以前就会，继续保持」。

In [1]:
! pip install -q -U transformers bitsandbytes accelerate

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"GPU可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU可用: True
GPU型号: Tesla T4
显存: 15.6 GB


# 加载模型

In [3]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    print("pad_token 为空，设置为 eos_token")
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("模型加载完成")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

模型加载完成


## 定义提示词（10类 × 3条 = 30条）

问题保持简短，回答在 150 token 以内即可覆盖。

In [4]:
# 10个话题类别，每类3个提示，共30条

PROMPTS = [
    # --- 1. 日常知识 ---
    "为什么天空是蓝色的？",
    "人一天需要喝多少水？",
    "为什么打哈欠会传染？",

    # --- 2. 科普常识 ---
    "光从太阳到地球需要多长时间？",
    "为什么夏天虫子特别多？",
    "月亮为什么有时候看起来特别大？",

    # --- 3. 历史文化 ---
    "中国的四大发明是哪四个？",
    "为什么春节要贴春联、放鞭炮？",
    "丝绸之路是什么？",

    # --- 4. 中文表达 ---
    "解释一下'塞翁失马，焉知非福'这句话",
    "用三个词描述一个秋天的傍晚",
    "'的地得'分别在什么情况下使用？",

    # --- 5. 生活技巧 ---
    "衣服上沾了红酒怎么快速处理？",
    "手机充电有什么保护电池的小技巧？",
    "眼睛疲劳时如何快速缓解？",

    # --- 6. 健康饮食 ---
    "早餐吃什么比较健康又方便？",
    "感冒初期应该怎么处理？",
    "久坐一天，有什么简单的拉伸动作推荐？",

    # --- 7. 职场学习 ---
    "番茄工作法是什么？怎么用？",
    "面试时被问'你最大的缺点'怎么回答比较好？",
    "如何快速记会议要点？",

    # --- 8. 情感人际 ---
    "朋友情绪低落时，我该怎么安慰他？",
    "如何礼貌地拒绝别人的请求？",
    "如何在新环境中快速融入团队？",

    # --- 9. 简单数学 ---
    "正方形面积是64平方厘米，周长是多少？",
    "鸡兔同笼：头共10个，脚共28只，鸡和兔各有几只？",
    "一个人花30元买了一只鸡，40元卖出，50元买回，60元卖出，赚了多少？",

    # --- 10. 创意趣味 ---
    "100元以内，送什么礼物能让朋友感到惊喜？",
    "如何把每天的通勤时间变得更有价值？",
    "推荐一件可以让日常生活更有趣的小事",
]

print(f"共 {len(PROMPTS)} 个提示词，涵盖 10 个类别")

共 30 个提示词，涵盖 10 个类别


# 批量生成回答

In [6]:
def generate_response(prompt: str, max_new_tokens: int = 512) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    return response.strip()


# 生成所有回答
replay_data = []
failed = []

print(f"开始生成，共 {len(PROMPTS)} 条...\n")

for i, prompt in enumerate(PROMPTS):
    try:
        response = generate_response(prompt)
        if len(response.strip()) < 5:
            raise ValueError(f"回答过短: {repr(response)}")

        replay_data.append({
            "conversations": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": response},
            ]
        })

        # 每5条打印一次进度
        if (i + 1) % 5 == 0 or i == 0:
            print(f"[{i+1}/{len(PROMPTS)}] Q: {prompt}")
            print(f"         A: {response[:80]}{'...' if len(response) > 80 else ''}")
            print()

    except Exception as e:
        print(f"[{i+1}] 失败: {prompt} 错误: {e}")
        failed.append(i)

print(f"\n生成完成：成功 {len(replay_data)} 条，失败 {len(failed)} 条")

开始生成，共 30 条...

[1/30] Q: 为什么天空是蓝色的？
         A: 天空之所以是蓝色的，是因为**瑞利散射**（Rayleigh scattering）现象。

我们来一步步解释这个过程：

### 1. 太阳光是什么颜色？
太...

[5/30] Q: 为什么夏天虫子特别多？
         A: 这是一个很有趣也很常见的问题！“为什么夏天虫子特别多？”其实背后有科学道理，我们可以从几个方面来解释：

---

### 1. **温度高，虫子繁殖快**
虫...

[10/30] Q: 解释一下'塞翁失马，焉知非福'这句话
         A: “塞翁失马，焉知非福”是一句中国古语，出自《淮南子·人间训》。它用来表达一种哲理性的思考：**坏事在某种情况下可能转为好事，好事也可能变成坏事，因此不能以一时的...

[15/30] Q: 眼睛疲劳时如何快速缓解？
         A: 眼睛疲劳时，可以通过以下几种方法快速缓解，帮助恢复眼部舒适度和视觉功能：

### 一、简单物理放松法（立即可用）
1. **闭眼休息或做“20-20-20”法...

[20/30] Q: 面试时被问'你最大的缺点'怎么回答比较好？
         A: 在面试中被问“你最大的缺点”时，关键在于**如何回答才能既显得真实、诚恳，又不损害你的竞争力**。一个优秀的回答应该：

✅ 展示自我认知  
✅ 与岗位要求相...

[25/30] Q: 正方形面积是64平方厘米，周长是多少？
         A: 我们知道正方形的面积是 64 平方厘米。

### 第一步：求边长

正方形面积公式是：

$$
A = a^2
$$

其中 $a$ 是边长。

所以：

$...

[30/30] Q: 推荐一件可以让日常生活更有趣的小事
         A: 当然可以！我为你推荐一件能让日常生活更有趣的小事：

✨“每天用不同颜色的笔写日记”✨

听起来简单，但其实能带来巨大的乐趣和新鲜感！

👉 做法：  
每天选...


生成完成：成功 30 条，失败 0 条


# 验证质量

In [7]:
# 抽查几条，人工确认质量
import random

samples = random.sample(replay_data, min(5, len(replay_data)))
print("=== 随机抽查 5 条 ===")
for s in samples:
    q = s["conversations"][0]["content"]
    a = s["conversations"][1]["content"]
    print(f"Q: {q}")
    print(f"A: {a[:200]}{'...' if len(a) > 200 else ''}")
    print("-" * 60)

=== 随机抽查 5 条 ===
Q: 如何礼貌地拒绝别人的请求？
A: 礼貌地拒绝别人的请求是维护人际关系、尊重他人感受同时又不失自我边界的重要能力。以下是一些实用且得体的拒绝方式，帮助你既保持真诚又避免让对方感到被冷落或受伤：

---

### 一、使用“温和+肯定”的开场
先肯定对方的请求或想法，再自然过渡到拒绝。

✅ 示例：
> “真的很感谢你想到我，这个想法/建议让我很感动！不过目前我的时间/精力/安排上确实有些紧张，恐怕无法参与/帮忙了。”

---

...
------------------------------------------------------------
Q: 早餐吃什么比较健康又方便？
A: 早餐是一天中最重要的一餐，它为身体提供能量，帮助启动新陈代谢。选择一份**健康又方便**的早餐，关键在于：营养均衡、低油低糖、易准备、易消化。

以下是一些既健康又方便的早餐建议，适合忙碌的生活节奏：

---

✅ 推荐搭配（健康 + 方便）：

1. **全谷物粥 / 燕麦粥 + 水果**
   - 例如：即食燕麦片 + 牛奶或植物奶 + 一颗香蕉或一小把蓝莓
   - 优点：富含膳食纤维、稳...
------------------------------------------------------------
Q: 一个人花30元买了一只鸡，40元卖出，50元买回，60元卖出，赚了多少？
A: 我们来一步一步分析这个题目：

题目是：

> 一个人花30元买了一只鸡，40元卖出，50元买回，60元卖出，赚了多少？

---

**第一步：第一次买卖**

- 花30元买进 → 成本：30元  
- 卖出40元 → 收入：40元  
→ 第一次赚了：40 - 30 = **10元**

---

**第二步：第二次买卖**

- 用50元买回 → 成本：50元  
- 卖出60元 → 收入...
------------------------------------------------------------
Q: 朋友情绪低落时，我该怎么安慰他？
A: 当朋友情绪低落时，真诚、耐心和共情的陪伴是最重要的。以下是一些具体而有效的安慰方式，可以帮助你的朋友感到被理解和支持：

---

### 1. **先倾听，不急于

# 保存数据

In [8]:
OUTPUT_FILE = "replay_data.json"

# 保存为和 train_data_augmented.json 兼容的格式
# system_prompt 留空，微调时不注入角色 prompt，让模型保持通用能力
output = {
    "system_prompt": "",
    "conversations": replay_data,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"已保存 {len(replay_data)} 条数据到 {OUTPUT_FILE}")
print(f"文件大小: {__import__('os').path.getsize(OUTPUT_FILE) / 1024:.1f} KB")

已保存 30 条数据到 replay_data.json
文件大小: 58.3 KB


In [9]:
# 下载到本地
from google.colab import files
files.download(OUTPUT_FILE)
print(f"已下载 {OUTPUT_FILE}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

已下载 replay_data.json
